# CMA-ME — Baseline

**Method:** CMA-ME (Covariance Matrix Adaptation MAP-Elites)  
**Seeds:** 2 (42, 123)  
**Task:** Ant-v5 Unidirectional Gait (4D Gait BD, 10×10×10×10 = 10000 bins)  
**Budget:** N_INIT_SAMPLES + N_EMITTERS × POPSIZE × N_STEPS ≈ 1e5

In [ ]:
# ============================================================
# Cell 1: Setup — mount Drive, clone repo, load SSLVE modules
# Run this cell once at the start of each Colab session.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!rm -rf org && git clone --filter=blob:none --sparse https://github.com/yugoguy/org.git
!cd org && git sparse-checkout set dev/SSLVE && git checkout Latent-Variable-Evolution
!pip install cma gymnasium[mujoco] -q

import sys, os, glob
sys.path.insert(0, 'org/dev/SSLVE')
for f in sorted(glob.glob('org/dev/SSLVE/*.py')):
    %run {f}

print('Setup complete.')

Mounted at /content/drive
Cloning into 'org'...
remote: Enumerating objects: 1903, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 1903 (delta 144), reused 34 (delta 34), pack-reused 1681 (from 1)
Receiving objects: 100% (1903/1903), 531.72 KiB | 5.32 MiB/s, done.
Resolving deltas: 100% (803/803), done.
remote: Enumerating objects: 1, done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 1 (from 1)
Receiving objects: 100% (1/1), 255 bytes | 255.00 KiB/s, done.
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 18 (delta 5), reused 1 (delta 1), pack-reused 6 (from 1)
Receiving objects: 100% (18/18), 72.87 KiB | 2.43 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Branch 'Latent-Variable-Evolution' set up to track remote branch 'Latent-Variable-Evolution' from 'origin'.
Switched to a new branch 'Latent-Variable

In [ ]:
# ============================================================
# Cell 2: Method Hyperparameters
# ============================================================
import numpy as np
import random
import torch

# --- Architecture ---
ARCHITECTURE     = [27, 64, 64, 8]  #@param
OUTPUT_ACTIVATION = 'tanh'  #@param {type:"string"}
MAX_STEPS        = 500   #@param {type:"integer"}
N_EPISODES       = 3     #@param {type:"integer"}
CTRL_COST_WEIGHT = 0.5   #@param {type:"number"}

# --- Gait BD Grid ---
BIN_SIZES        = [10, 10, 10, 10]  #@param

# --- Fitness ---
TOP_K            = 1     #@param {type:"integer"}
MAX_FITNESS      = 500.0 #@param {type:"number"}
GREEDY_MEM       = True  #@param {type:"boolean"}

# --- CMA-ME ---
N_EMITTERS       = 10    #@param {type:"integer"}
SIGMA_INIT       = 0.05  #@param {type:"number"}
POPSIZE          = 50    #@param {type:"integer"}
N_INIT_SAMPLES   = 500   #@param {type:"integer"}
SEPARABLE        = True  #@param {type:"boolean"}

# --- Search budget ---
N_STEPS          = 200   #@param {type:"integer"}

# --- Checkpoint base directory (saved to Google Drive) ---
CKPT_BASE_DIR    = '/content/drive/MyDrive/shared_ckpts/ant_gait/Ant_CMAME/'  #@param {type:"string"}

total_evals = N_INIT_SAMPLES + N_EMITTERS * POPSIZE * N_STEPS
print(f'Method: CMA-ME')
print(f'Architecture: {ARCHITECTURE}, Output: {OUTPUT_ACTIVATION}')
print(f'Gait BD: {BIN_SIZES} ({np.prod(BIN_SIZES)} bins)')
print(f'N_EMITTERS={N_EMITTERS}, POPSIZE={POPSIZE}, SIGMA_INIT={SIGMA_INIT}')
print(f'N_INIT_SAMPLES={N_INIT_SAMPLES}, N_STEPS={N_STEPS}')
print(f'Total evaluations: {total_evals}')
print(f'Checkpoints will be saved to: {CKPT_BASE_DIR}')

Method: CMA-ME
Architecture: [27, 64, 64, 8], Output: tanh
Gait BD: [10, 10, 10, 10] (10000 bins)
N_EMITTERS=10, POPSIZE=50, SIGMA_INIT=0.05
N_INIT_SAMPLES=500, N_STEPS=200
Total evaluations: 100500
Checkpoints will be saved to: /content/drive/MyDrive/shared_ckpts/ant_gait/Ant_CMAME/


In [ ]:
# ============================================================
# Cell 3: Run All Experiments (2 seeds)
# Each run saves a checkpoint + plots immediately after finishing.
# ============================================================

SEEDS = [42, 123]

fitness_fn = lambda info: -info['forward_sum']

all_results = []

weight_dim = MLP_Agent(ARCHITECTURE, output_activation=OUTPUT_ACTIVATION).get_weight_dim()
print(f'Weight dim: {weight_dim}')

for seed in SEEDS:
    print(f'\n========== seed={seed} ==========')

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    collector = AntOmniCollector(
        max_steps=MAX_STEPS,
        n_episodes=N_EPISODES,
        ctrl_cost_weight=CTRL_COST_WEIGHT,
        seed=seed,
    )
    bd = AntGaitBD(bin_sizes=BIN_SIZES)
    bm = MAPElitesBM(behavior_descriptor=bd, fitness_fn=fitness_fn, top_k=TOP_K, max_fitness=MAX_FITNESS)

    orchestrator = CMAME(
        agent_class=MLP_Agent,
        architecture=ARCHITECTURE,
        agent_kwargs={'output_activation': OUTPUT_ACTIVATION},
        collector=collector,
        behavior_matching=bm,
        n_emitters=N_EMITTERS,
        sigma_init=SIGMA_INIT,
        popsize=POPSIZE,
        greedy_mem=GREEDY_MEM,
        separable=SEPARABLE,
        n_init_samples=N_INIT_SAMPLES,
    )

    orchestrator.run(n_steps=N_STEPS)

    # Save checkpoint immediately after this run
    ckpt_path = os.path.join(
        CKPT_BASE_DIR,
        f'CMAME_emit{N_EMITTERS}_pop{POPSIZE}_sigma{SIGMA_INIT}_seed{seed}/'
    )
    save_cmame_checkpoint(ckpt_path, orchestrator)
    print(f'Checkpoint saved: {ckpt_path}')

    # Save plots into checkpoint folder
    orchestrator.plot_history(save_path=ckpt_path + 'plot_history.png')
    print(f'Plots saved to: {ckpt_path}')

    # Force Drive sync after each run
    from google.colab import drive
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print('Drive re-synced.')

    # Print results
    f_min, f_mean, f_max = bm.fitness_stats()
    qd  = bm.qd_score()
    cov = bm.coverage()
    print(f'QD-score: {qd:.4f} | Coverage: {cov:.4f} | Best fitness: {f_min:.6f}')
    print(f'Total evaluations: {orchestrator.total_evals}')

    all_results.append({
        'seed': seed,
        'qd_score': qd, 'coverage': cov,
    })

print('\n====== All runs complete ======')

Weight dim: 6472

========== seed=42 ==========

--- CMAME Step 1/200 ---
Init: 500/500 [167s elapsed, 0s remaining]
Archive: 128, Bins: 128, Coverage: 0.0128, Fitness min/mean/max: -65.40/-5.71/74.24, QD-score: 64730.8806, Evals: 500

--- CMAME Step 2/200 ---
Archive: 274, Bins: 274, Coverage: 0.0274, Fitness min/mean/max: -65.40/-3.87/74.24, QD-score: 138060.1101, Evals: 1000

--- CMAME Step 3/200 ---
Archive: 385, Bins: 385, Coverage: 0.0385, Fitness min/mean/max: -65.40/-3.98/74.24, QD-score: 194032.3564, Evals: 1500

--- CMAME Step 4/200 ---
Archive: 501, Bins: 501, Coverage: 0.0501, Fitness min/mean/max: -65.40/-4.31/74.24, QD-score: 252656.9472, Evals: 2000

--- CMAME Step 5/200 ---
Archive: 589, Bins: 589, Coverage: 0.0589, Fitness min/mean/max: -65.40/-5.16/74.24, QD-score: 297538.9648, Evals: 2500

--- CMAME Step 6/200 ---
Archive: 692, Bins: 692, Coverage: 0.0692, Fitness min/mean/max: -133.37/-6.04/74.24, QD-score: 350178.1430, Evals: 3000

--- CMAME Step 7/200 ---
Archive: